# Reproduce TurboQuant StorageManager round trips

Run all cells from any directory in an LMCache checkout. This invokes the checked-in Mock-L2 and FS-L2 round-trip tests for all four TurboQuant presets and fails if CUDA tests are skipped or any case fails.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
from pathlib import Path
import os
import subprocess
import sys

import torch

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "lmcache").is_dir()
)
TEST_FILE = ROOT / "tests/v1/distributed/serde/test_turboquant.py"
assert TEST_FILE.is_file(), TEST_FILE
assert torch.cuda.is_available(), "CUDA is required"
print(
    {
        "root": str(ROOT),
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
    }
)

In [ ]:
# SPDX-License-Identifier: Apache-2.0
selection = (
    "test_turboquant_storage_manager_roundtrip "
    "or test_turboquant_fs_storage_manager_roundtrip"
)
command = [
    sys.executable,
    "-m",
    "pytest",
    "-q",
    "-s",
    str(TEST_FILE),
    "-k",
    selection,
]
env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT) + os.pathsep + env.get("PYTHONPATH", "")
print("Running:", " ".join(command))
completed = subprocess.run(
    command,
    cwd=ROOT,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(completed.stdout)
completed.check_returncode()
assert "8 passed" in completed.stdout, "expected all 8 CUDA round-trip cases to run"
print({"status": "passed", "cuda_roundtrips": 8})